# How to set up and test methods: QM8 

This file shows how the system works and allows you to check that the datasets are being loaded correctly and are in the correct formats. 

Run this with a subset of the data you plan to test (say the test set) only to see if it works. If it does, then run the python script to set up and save all the topological data as a .hdf5 file, then you only have to create the dataset once.

## Method Followed:

1. Use dummy featurizer or my dummy featurizer to load the molecule information from the `.sdf` file into RDKit objects
2. Create topological feature dataset from dataset
3. Check that things are in the correct format for ML to work (we got random forest and a keras FFNN model). 

Extra stuff:
4. code to look at the data and interrogate the database
5. comparisons to deepchems model on this dataset. 


In [1]:
import sys
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import deepchem as dc

import tensorflow as tf
import os
import sys
import rdkit
import h5py
import helper_functions as h

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.tri
import rdkit.Chem
import rdkit.Chem.AllChem as Chem
import rdkit.Chem.AllChem as AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import rdMolDescriptors
import mpl_toolkits.mplot3d
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from collections import Counter

print("TensorFlow version: " + tf.__version__)

# topology stuff
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
from gtda.diagrams import PersistenceEntropy
from gtda.diagrams import NumberOfPoints
from gtda.diagrams import Amplitude
from sklearn.pipeline import make_union, Pipeline

# fixc this at some point
sys.path.append(r"C:\Users\ella_\Documents\GitHub\graphs_and_topology_for_chemistry")
sys.path.append(r"C:\Users\ella_\Documents\GitHub\icosahedron_projection")

import projection
from projection.molecule import Molecule
from projection.pdbmolecule import PDBMolecule
from projection.mol2molecule import Mol2Molecule

import helper_functions as h
#from projection.face import Face

# $UN THIS
save_dir=r'F:\Nextcloud\science\Datasets\converted_pdbbind\v2015'
data_dir=r'F:\Nextcloud\science\Datasets'
results_dir=r"F:\Nextcloud\science\results\topology_and_graphs\QM8"
test_file='qm8.csv'
out_file_name='qm8_topological_features.hdf5'
make_dataset=False # whether to recalc the dataset

name_file_name="INDEX_core_name.2013",
data_file_name="INDEX_core_data.2013",
cluster_file_name = "INDEX_core_cluster.2013"
print(f"DeepChem version: {dc.__version__}")

RuntimeError: Failed to import transformers.models.bert because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
No module named 'flatbuffers'

# 1. Load data with no featurization

In [ ]:
# This loads the data without doing any featurization
tasks, datasets, transformers = dc.molnet.load_qm8(
    shard_size=2000, featurizer=h.My_Dummy_Featurizer(None), splitter="random")

In [ ]:
train_dataset, valid_dataset, test_dataset = datasets

In [ ]:
# we're only going to use a small dataset for testing the code here!
my_dataset=test_dataset
num_of_molecules = len(test_dataset)
orig_test_dataset = test_dataset

In [ ]:
num_of_molecules = 10

# 2. Create topological features dataset

This makes the topological features.

In [ ]:
topol_feat_list, topol_feat_mat = h.make_topological_features_from_deepchem(my_dataset,
                                          file_type='dc',
                                          verbose=False,
                                          num_of_molecules = num_of_molecules,
                                          num_of_features = 18,
                                          data_dir='',
                                          do_specified_range=False,
                                          selected_range=[])

We use the np.array version of topol as **X**. And make a nice dataframe for **y**.

In [ ]:
my_dataset.y

In [ ]:
h.copy_targets_into_csv(
                y_fh=y_fh,
                my_dataset=untransformed_train_y
            )

In [ ]:
tasks, datasets, transformers = dc.molnet.load_qm8(
    shard_size=2000, 
    featurizer=h.My_Dummy_Featurizer(None), 
    splitter="index")

In [ ]:
train_dataset, valid_dataset, test_dataset = datasets
test_dataset.X[-1]

In [ ]:
train_dataset.ids

In [ ]:
train_dataset.y[3]
full_targets = [x[0] for x in transformers[0].untransform(train_dataset.y)] \
+ [x[0] for x in transformers[0].untransform(valid_dataset.y)] \
+ [x[0] for x in transformers[0].untransform(test_dataset.y)]

In [ ]:
import csv

## we have to do this as the sdf file is missing 4 datapoints, so we need all the data
## to be taken directly from the deepchem dataset

# open the file in the write mode
f = open(os.path.join(save_dir,'qm8_SMILES.csv'), 'w',newline='')

# create the csv writer
writer = csv.writer(f)

In [ ]:
[x for x in transformers[0].untransform(dataset.y)][0]

In [ ]:


for molecule_num in range(10):
    row = [dataset.ids[molecule_num]]
    print(row)
    # write a row to the csv file
    writer.writerow(row)
f.close()

In [ ]:


dataset=train_dataset
un_y = [x[0] for x in transformers[0].untransform(dataset.y)]
for molecule_num in range(len(dataset)):
    row = [dataset.ids[molecule_num]]
    #print(row)
    # write a row to the csv file
    writer.writerow(row)

dataset=valid_dataset
un_y = [x[0] for x in transformers[0].untransform(dataset.y)]
for molecule_num in range(len(dataset)):
    row = [dataset.ids[molecule_num]]
    #print(row)
    # write a row to the csv file
    writer.writerow(row)
    
dataset=test_dataset
un_y = [x[0] for x in transformers[0].untransform(dataset.y)]
for molecule_num in range(len(dataset)):
    row = [dataset.ids[molecule_num]]
    #print(row)
    # write a row to the csv file
    writer.writerow(row)

    # close the file
f.close()


In [ ]:
X_data = topol_feat_mat
y_data = pd.DataFrame(my_dataset.y, columns=tasks)

In [ ]:
dataset.tasks

# Check it all works with a random forest

In [ ]:
(train_scores,test_scores) = h.run_repeated_RF_tests(
    X_data,
    y_data,
    num_of_repeats=10,
    num_of_estimators=100,
    test_set_size=10,
    validate_set_size=10)

out=h.nice_stats_outputter(
    train_scores, 
    test_scores, 
    validate_scores='')
(min_train_RF_large, mean_train_RF_large, std_error_train_RF_large, max_train_RF_large,
                min_test_RF_large, mean_test_RF_large, std_error_test_RF_large, max_test_RF_large)=out
RF_large_train_scores=train_scores
RF_large_test_scores=test_scores

In [ ]:
results_list = []

In [ ]:
egg = ['Rf'] + [x for x in out][:4]
egg = egg + [0,0,0,0] # as no validate
egg = egg + [x for x in out][4:]
results_list.append(egg)
egg

## Run with my keras model tests

NTS you may want to change this function so that it returns the validate stats as well!

In [ ]:
keras_model = h.create_keras_model()
out=h.run_repeated_keras_NN_tests(
    X_data,
    y_data,
    keras_model=None,
    num_of_repeats=5,
    num_of_epochs=100,
    test_set_size=int(num_of_molecules*0.1),
    validate_set_size=int(num_of_molecules*0.1))

(train_scores_r2, test_scores_r2, train_scores_rmse, test_scores_rmse) = out

In [ ]:
results_dataframe = pd.DataFrame(columns=['Exp name', 
                        'train_min', 'train_mean', 'train_std', 'train_max',
                        'valid_min', 'valid_mean', 'valid_std', 'valid_max',
                        'test_min', 'test_mean', 'test_std', 'test_max',])

In [ ]:
out=h.nice_stats_outputter(
    train_scores_rmse, 
    test_scores_rmse, 
    validate_scores='')
(min_train_keras, mean_train_keras, std_train_error_keras, max_train_keras,
                min_test_keras, mean_test_keras, std_error_test_keras, max_test_keras)=out
#RF_large_train_scores=train_scores
#RF_large_test_scores=test_scores

In [ ]:
egg = ['keras'] + [x for x in out][:4]
egg = egg + [0,0,0,0] # as no validate
egg = egg + [x for x in out][4:]
results_list.append(egg)
egg

## Using DeepChem's Multitask regressor

We have to create new dataset in the format that DeepChem expects.

In [ ]:
my_dataset = dc.data.NumpyDataset(X=X_data, y=y_data)
print(my_dataset)

In [ ]:
my_dataset.to_dataframe().head()

In [ ]:
# saving to disc
#import tempfile

#with tempfile.TemporaryDirectory() as data_dir:
#    disk_dataset = dc.data.DiskDataset.from_numpy(X=X, y=y, data_dir=data_dir)
#    print(disk_dataset)

We can use DeepChem's splitter. You need to instantiate it as an object first.

In [ ]:
Splitter_Object = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = Splitter_Object.train_valid_test_split(dataset=my_dataset,
                                               frac_train=0.8,
                                               frac_valid=0.1,
                                               frac_test=0.1)

In [ ]:
len(my_dataset.X[0])

For this problem we're using the RMSE which is averaged across all tasks.

This bit of optimises the hyperparameters by running a few epochs of several models.

In [ ]:
params_dict = {
    'n_tasks': [len(tasks)],
    'n_features': [len(my_dataset.X[0])],
    'layer_sizes': [[500], [1000], [1000, 1000]],
    'dropouts': [0.2, 0.5],
    'learning_rate': [0.001, 0.0001]
}
optimizer = dc.hyper.GridHyperparamOpt(dc.models.MultitaskRegressor)
#metric = dc.metrics.Metric(dc.metrics.rms_score)
# We want to know the RMS, averaged across tasks
metric = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
best_model, best_hyperparams, all_results = optimizer.hyperparam_search(
        params_dict, train_dataset, valid_dataset, metric, transformers)

In [ ]:
all_results

In [ ]:
best_hyperparams

Now we copy the hyperparameters in and run the model with early stopping. 

In [ ]:
train_scores, validate_scores, test_scores = [],[],[]
for i in range(5):
    model = dc.models.MultitaskRegressor(n_tasks=len(tasks),
                                          n_features=18,
                                          layer_sizes=[1000],
                                          dropouts=0.5,
                                          learning_rate=0.001)
    callback = dc.models.ValidationCallback(valid_dataset, 1000, metric)
    model.fit(train_dataset, nb_epoch=50, callbacks=callback)

    # line below returns a dictionary
    train_scores.append(model.evaluate(train_dataset, metric, transformers=transformers)['mean-rms_score'])
    validate_scores.append(model.evaluate(valid_dataset, metric, transformers=transformers)['mean-rms_score'])
    test_scores.append(model.evaluate(test_dataset, metric, transformers=transformers)['mean-rms_score'])

In [ ]:
model.evaluate(train_dataset, metric, transformers=transformers)

In [ ]:
out=h.nice_stats_outputter(
    train_scores=train_scores, 
    test_scores=test_scores, 
    validate_scores=validate_scores)


In [ ]:
egg = ['topol_f'] + [x for x in out]
#egg
#egg = egg + [0,0,0,0] # as no validate
#egg = egg + [x for x in out][4:]
results_list.append(egg)
egg

# 4. (Extra) Look at the molecules and plot topology stuff

In [ ]:
# this should grab a random molecule
mol = test_dataset.X[0]
mol

In [ ]:

print(f"Dataset has {num_of_molecules} molecules")
num_of_features = 18

In [ ]:
coords = mol.GetConformer().GetPositions()

In [ ]:
# Testing the topology stuff - this gives a nice point cloud picture
plot_point_cloud(coords)

In [ ]:
## does the topology stuff
# Track connected components, loops, and voids
homology_dimensions = [0, 1, 2]

# Collapse edges to speed up H2 persistence calculation!
persistence = VietorisRipsPersistence(
    metric="euclidean",
    homology_dimensions=homology_dimensions,
    n_jobs=6,
    collapse_edges=True,
)
reshaped_coords=coords[None, :, :]
diagrams_basic = persistence.fit_transform(reshaped_coords)

In [ ]:
# makes a nice birth-death diagram
plot_diagram(diagrams_basic[0])

In [ ]:
train_dataset, valid_dataset, test_dataset = datasets
len(tasks)
f'Compound train/valid/test split: {len(train_dataset)}/{len(valid_dataset)}/{len(test_dataset)}'

In [ ]:
train_dataset.X

# 5. DeepChem's models (no topology)

In [ ]:
tasks, datasets, transformers = dc.molnet.load_qm8(
    shard_size=2000, featurizer="ECFP", splitter="random")
_, _, small_dataset = datasets
## we're jsut going to use the test dataset here to test the code
splitter = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(
  dataset=small_dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1
)

# We want to know the RMS, averaged across tasks
avg_rms = dc.metrics.Metric(dc.metrics.rms_score, np.mean)

In [ ]:
params_dict = {
    'n_tasks': [len(tasks)],
    'n_features': [1024],
    'layer_sizes': [[500], [1000], [1000, 1000]],
    'dropouts': [0.2, 0.5],
    'learning_rate': [0.001, 0.0001]
}
optimizer = dc.hyper.GridHyperparamOpt(dc.models.MultitaskRegressor)
#metric = dc.metrics.Metric(dc.metrics.rms_score)
metric = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
best_model, best_hyperparams, all_results = optimizer.hyperparam_search(
        params_dict, train_dataset, valid_dataset, metric, transformers)

In [ ]:
best_hyperparams

In [ ]:
train_scores, validate_scores, test_scores = [],[],[]
for i in range(5):
    model = dc.models.MultitaskRegressor(n_tasks=len(tasks),
                                          n_features=1024,
                                          layer_sizes=[1000],
                                          dropouts=0.2,
                                          learning_rate=0.001)
    callback = dc.models.ValidationCallback(valid_dataset, 1000, metric)
    model.fit(train_dataset, nb_epoch=50, callbacks=callback)

    # line below returns a dictionary
    train_scores.append(model.evaluate(train_dataset, metric, transformers=transformers)['mean-rms_score'])
    validate_scores.append(model.evaluate(valid_dataset, metric, transformers=transformers)['mean-rms_score'])
    test_scores.append(model.evaluate(test_dataset, metric, transformers=transformers)['mean-rms_score'])

In [ ]:
model.evaluate(train_dataset, metric, transformers=transformers)

In [ ]:
out=h.nice_stats_outputter(
    train_scores=train_scores, 
    test_scores=test_scores_rmse, 
    validate_scores=validate_scores)

In [ ]:
egg = ['ECFP'] + [x for x in out]
#egg
#egg = egg + [0,0,0,0] # as no validate
#egg = egg + [x for x in out][4:]
results_list.append(egg)
egg

In [ ]:
n_layers = 3
model = dc.models.MultitaskRegressor(
    len(tasks),
    n_features=1024,
    layer_sizes=[1000] * n_layers,
    dropouts=[.25] * n_layers,
    weight_init_stddevs=[.02] * n_layers,
    bias_init_consts=[1.] * n_layers,
    learning_rate=.0003,
    weight_decay_penalty=.0001,
    batch_size=100)

In [ ]:
model.fit(train_dataset, nb_epoch=5)

In [ ]:
# We now evaluate our fitted model on our training and validation sets
train_scores = model.evaluate(train_dataset, [avg_rms], transformers)
assert train_scores['mean-rms_score'] < 10.00
valid_scores = model.evaluate(valid_dataset, [avg_rms], transformers)
assert valid_scores['mean-rms_score'] < 10.00
print(valid_scores)
test_scores = model.evaluate(test_dataset, [avg_rms], transformers)
assert test_scores['mean-rms_score'] < 10.00
print(f"Train scores average = {train_scores}")
print(f"Validate scores average = {valid_scores}")
print(f"Test scores average = {test_scores}")

In [ ]:
train_scores

### all data for comparison
this in trained on the entire dataset, so should be much better

In [ ]:
tasks, datasets, transformers = dc.molnet.load_qm8(
    shard_size=2000, featurizer="ECFP", splitter="random")
# We want to know the RMS, averaged across tasks
avg_rms = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
train_dataset, valid_dataset, test_dataset = datasets

n_layers = 3
model = dc.models.MultitaskRegressor(
    len(tasks),
    n_features=1024,
    layer_sizes=[1000] * n_layers,
    dropouts=[.25] * n_layers,
    weight_init_stddevs=[.02] * n_layers,
    bias_init_consts=[1.] * n_layers,
    learning_rate=.0003,
    weight_decay_penalty=.0001,
    batch_size=100)

model.fit(train_dataset, nb_epoch=5)

# We now evaluate our fitted model on our training and validation sets
train_scores = model.evaluate(train_dataset, [avg_rms], transformers)
assert train_scores['mean-rms_score'] < 10.00
valid_scores = model.evaluate(valid_dataset, [avg_rms], transformers)
assert valid_scores['mean-rms_score'] < 10.00
print(valid_scores)
test_scores = model.evaluate(test_dataset, [avg_rms], transformers)
assert test_scores['mean-rms_score'] < 10.00
print(f"Train scores average = {train_scores}")
print(f"Validate scores average = {valid_scores}")
print(f"Test scores average = {test_scores}")

# Results

(I ran keras twice somehow).

In [ ]:
results_dataframe = pd.DataFrame(results_list, columns=['Exp name', 
                        'train_min', 'train_mean', 'train_std', 'train_max',
                        'valid_min', 'valid_mean', 'valid_std', 'valid_max',
                        'test_min', 'test_mean', 'test_std', 'test_max'])

In [ ]:
results_dataframe

In [ ]:
means = []
stds = []
for i in range(len(results_list)):
    means.append(results_list[i][2])
    means.append(results_list[i][10])
    
stds = []
for i in range(len(results_list)):
    stds.append(results_list[i][3])
    stds.append(results_list[i][11])

In [ ]:
labels = results_dataframe['Exp name']
#labels[]
labels = [x for x in labels]

In [ ]:
labels

In [ ]:
labels = ['Rf', 'keras', 'topol_f', 'ECFP']

In [ ]:
Best=0.053

#data=[dc_NN_MR_small_train_scores_rmse,
#      dc_NN_MR_small_test_scores_rmse,
#      dc_NN_MR_large_train_scores_rmse,
#      dc_NN_MR_large_test_scores_rmse]
      

#x_positions = [1,2,3,4]
# train test
#means=[mean_train_dc_NN_MR_small_rmse, mean_test_dc_NN_MR_small_rmse, mean_train_dc_NN_MR_large_rmse, mean_test_dc_NN_MR_large_rmse]
#stds=[std_error_train_dc_NN_MR_small_rmse, std_error_test_dc_NN_MR_small_rmse, std_error_train_dc_NN_MR_large_rmse, std_error_test_dc_NN_MR_large_rmse]



x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
#plt.fontsize(14)
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728","#1f77f4","#f62728"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
#for i in range(4):
#    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")

plt.plot([0.5,max(x_positions) + 0.5],[Best,Best],'k',linewidth=4,linestyle='-.')
axes=plt.gca()
#axes.set_ylim([0.5,4.5])
#axes.set_xlim([0.45,4.55])
plt.xticks([1.5,3.5,5.5,7.5],labels)
plt.ylabel('RMSE kcal/mol')
plt.savefig(os.path.join(results_dir, "2021_08_06_QM8_rmse_all_tests_run2.png"))

## Write up

We use the mean RMSE across all tasks in this work.

## Finding 1. 

Random forest baseline shows that the problem is difficult. The keras model uses the topological features but it not optmised, so gives a benchmark of no optimisation. 

The ECFP was optimised. So using topological features on a naive and unoptimised NN is the same as optimising with ECFP features. 

In [ ]:
Best=0.053

#data=[dc_NN_MR_small_train_scores_rmse,
#      dc_NN_MR_small_test_scores_rmse,
#      dc_NN_MR_large_train_scores_rmse,
#      dc_NN_MR_large_test_scores_rmse]
      

#x_positions = [1,2,3,4]
# train test
#means=[mean_train_dc_NN_MR_small_rmse, mean_test_dc_NN_MR_small_rmse, mean_train_dc_NN_MR_large_rmse, mean_test_dc_NN_MR_large_rmse]
#stds=[std_error_train_dc_NN_MR_small_rmse, std_error_test_dc_NN_MR_small_rmse, std_error_train_dc_NN_MR_large_rmse, std_error_test_dc_NN_MR_large_rmse]

#means = means[:-4]
#stds = stds[:-4]


x_positions = [x+1 for x in range(len(means))]
plt.figure(figsize=(16, 9))
#plt.fontsize(14)
plt.rcParams.update({'font.size': 22})

plt.bar(x_positions, means, 
        color=["#1f77f4","#f62728","#1f77f4","#f62728"], 
        edgecolor='k',
        linewidth='3',
        yerr=stds, 
        error_kw=dict(lw=5, capsize=15, capthick=3))
#for i in range(4):
#    x=data[i]; plt.plot(np.ones(len(x))*(i+1),x,'o',color="#444444")

plt.plot([0.5,max(x_positions) + 0.5],[Best,Best],'k',linewidth=4,linestyle='-.')
axes=plt.gca()
axes.set_ylim([-0.2,1.2])
plt.xticks([1.5,3.5,5.5,7.5],['','','topological features','ECFP'])
axes.set_xlim([4.5,8.5])
plt.ylabel('Mean RMSE kcal/mol')
plt.savefig(os.path.join(results_dir, "2021_08_06_QM8_rmse_run2.png"))

## Finding 2.

topol and ECFP results above were training on the same type of NN and both sets of hyperparameters were optimised.

The black dot-dashed line is at 0.053 which is the average RMSE error from using the entire dataset (admittedly no optimisation but life is too short) and ECFPs. 

The bars are train and test from hyperparameter optimised runs using topological features (size 18) or ECFP (size 1024).

**Firstly:** topol features beats ECFP on this dataset

**Secondly** topol features is as good (on this test dataset) as a model trained on the larger dataset! N.B. this dataset is 1/10th of the size of the larger dataset. 

## Graph Conv on QM8

broken from here on!

In [ ]:
tasks, datasets, transformers = dc.molnet.load_qm8(
   shard_size=2000, featurizer="GraphConv", split="random")
_, _, my_dataset = datasets
## we're jsut going to use the test dataset here to test the code
splitter = dc.splits.RandomSplitter()
train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(
  dataset=my_dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1
)

# RMS, averaged across tasks
avg_rms = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
model = dc.models.GraphConvModel(
   len(tasks), batch_size=128, mode='regression')
# Fit trained model
model.fit(train_dataset, nb_epoch=5)
# We now evaluate our fitted model on our training and validation sets
train_scores = model.evaluate(train_dataset, [avg_rms], transformers)
assert train_scores['mean-rms_score'] < 10.00

valid_scores = model.evaluate(valid_dataset, [avg_rms], transformers)
assert valid_scores['mean-rms_score'] < 10.00

print(f"Train scores average = {train_scores}")
print(f"Validate scores average = {valid_scores}")
print(f"Test scores average = {test_scores}")

In [ ]:
splitter = dc.splits.RandomSplitter()
tasks, datasets, transformers = dc.molnet.load_qm8(
   shard_size=2000, featurizer="GraphConv", splitter="random")
train_dataset, valid_dataset, test_dataset = datasets

#train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(
#  dataset=my_dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1
#)
# RMS, averaged across tasks
avg_rms = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
model = dc.models.GraphConvModel(
   len(tasks), batch_size=128, mode='regression')
# Fit trained model
model.fit(train_dataset, nb_epoch=5)
# We now evaluate our fitted model on our training and validation sets
train_scores = model.evaluate(train_dataset, [avg_rms], transformers)
assert train_scores['mean-rms_score'] < 10.00

valid_scores = model.evaluate(valid_dataset, [avg_rms], transformers)
assert valid_scores['mean-rms_score'] < 10.00

print(f"Train scores average = {train_scores}")
print(f"Validate scores average = {valid_scores}")
print(f"Test scores average = {test_scores}")

In [ ]:
splitter = dc.splits.RandomSplitter()
tasks, datasets, transformers = dc.molnet.load_qm8(
   shard_size=2000, featurizer="GraphConv", splitter="random")
_, _, my_dataset = datasets

train_dataset, valid_dataset, test_dataset = splitter.train_valid_test_split(
  dataset=my_dataset, frac_train=0.8, frac_valid=0.1, frac_test=0.1
)
# RMS, averaged across tasks
avg_rms = dc.metrics.Metric(dc.metrics.rms_score, np.mean)
model = dc.models.GraphConvModel(
   len(tasks), batch_size=128, mode='regression')
# Fit trained model
model.fit(train_dataset, nb_epoch=5)
# We now evaluate our fitted model on our training and validation sets
train_scores = model.evaluate(train_dataset, [avg_rms], transformers)
assert train_scores['mean-rms_score'] < 10.00

valid_scores = model.evaluate(valid_dataset, [avg_rms], transformers)
assert valid_scores['mean-rms_score'] < 10.00

print(f"Train scores average = {train_scores}")
print(f"Validate scores average = {valid_scores}")
print(f"Test scores average = {test_scores}")

In [ ]:
train_dataset